# Rule based clone generation

## Generating

In [8]:
import ast
import random


class RenameVariable(ast.NodeTransformer):
    def __init__(self):
        self.mapping = {}

    def visit_FunctionDef(self, node):
        for arg in node.args.args:
            new_name = f"var_{random.randint(1000,9999)}"
            self.mapping[arg.arg] = new_name
            arg.arg = new_name
        self.generic_visit(node)
        return node

    def visit_Name(self, node):
        if isinstance(node.ctx, (ast.Store, ast.Load)) and node.id in self.mapping:
            node.id = self.mapping[node.id]
        return node
 


class ArithmeticTransformer(ast.NodeTransformer):
    def visit_BinOp(self, node):
        self.generic_visit(node)
        if isinstance(node.op, ast.Add):
            node.left, node.right = node.right, node.left
        elif isinstance(node.op, ast.Sub):
            node.left, node.right = node.right, node.left
            node.op = ast.Sub()
        return node

class InlineConstants(ast.NodeTransformer):
    def __init__(self):
        self.constants = {}

    def visit_Assign(self, node):
        if isinstance(node.value, (ast.Constant, ast.Num, ast.Str)):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    self.constants[target.id] = node.value
        return node

    def visit_Name(self, node):
        if node.id in self.constants:
            return self.constants[node.id]
        return node


class InsertPass(ast.NodeTransformer):
    def visit_FunctionDef(self, node):
        new_body = []
        for stmt in node.body:
            new_body.append(stmt)
            if random.random() < 0.3:
                new_body.append(ast.Pass())
        node.body = new_body
        return node


## Validating with Tests

In [9]:
import textwrap
import tempfile
import subprocess
import sys
import os

def validate_clone(clone_code: str, tests: list) -> bool:
    """
    Validate a clone by writing it and its tests to a temporary .py file,
    then running it as a script to execute all unittest.TestCase classes.
    """
    try:
        # Dedent clone code and test code
        clone_code_dedented = textwrap.dedent(clone_code)
        tests_dedented = "\n".join([textwrap.dedent(tc) for tc in tests])

        # Combine into a full Python script
        full_script = f"""
{clone_code_dedented}

{tests_dedented}

if __name__ == "__main__":
    import unittest
    unittest.main()
"""

        # Write to a temporary file
        with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
            f.write(full_script)
            temp_filename = f.name

        # Run the temp file in a subprocess
        result = subprocess.run(
            [sys.executable, temp_filename],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        # Clean up temp file
        os.remove(temp_filename)

        if result.returncode == 0:
            return True
        else:
            print("Unit test failed:")
            print(result.stdout)
            print(result.stderr)
            return False

    except Exception as e:
        print("Validation error:", e)
        return False


## Update dataset

In [11]:
import ast
import astor
import textwrap
import json

# --- Transformation classes ---
transformations = [
    ("RenameVariable", RenameVariable), 
    ("ArithmeticTransformer", ArithmeticTransformer), 
    ("InsertPass", InsertPass)
]

NUM_CLONES_PER_TRANSFORMATION = 2

# --- Load your dataset ---
with open("../dataset/bigcodebench_normalized.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

dataset_sample = dataset[:4]

# --- Generate clones for each entry ---
for idx, entry in enumerate(dataset_sample):
    print(f"\n=== Processing entry {idx+1}/{len(dataset_sample)} ===")
    
    code = entry["original_code"]
    tests = entry["test"]
    
    clones_list = []
    
    for trans_name, TransClass in transformations:
        for i in range(NUM_CLONES_PER_TRANSFORMATION):
            try:
                tree = ast.parse(textwrap.dedent(code))
                clone_tree = TransClass().visit(tree)
                ast.fix_missing_locations(clone_tree)
                clone_code = astor.to_source(clone_tree)

                # Validate clone using your existing validate_clone function
                if validate_clone(clone_code, tests):
                    print(f"[{trans_name}] Clone {i+1} ✅ Passed")
                    clones_list.append({
                    "transformation": trans_name,
                    "code": clone_code})
                else:
                    print(f"[{trans_name}] Clone {i+1} ❌ Failed")
            except Exception as e:
                print(f"Error generating clone for {trans_name}: {e}")
    
    entry["clones"] = clones_list

with open("../results/bigcodebench_clones.json", "w", encoding="utf-8") as f:
    json.dump(dataset_sample, f, indent=2)

print("\nFinished generating clones for the dataset.")



=== Processing entry 1/4 ===
[RenameVariable] Clone 1 ✅ Passed
[RenameVariable] Clone 2 ✅ Passed
Unit test failed:

FFE.FFFF.F
ERROR: test_empty_list (__main__.TestCases.test_empty_list)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\lamp6\AppData\Local\Temp\tmpji9x9zvl.py", line 45, in test_empty_list
    result = task_func([])
  File "C:\Users\lamp6\AppData\Local\Temp\tmpji9x9zvl.py", line 12, in task_func
    diffs = [abs(perm[1 + i] - perm[i]) for i in range(1 - len(perm))]
                 ~~~~^^^^^^^
IndexError: list index out of range

FAIL: test_custom_list (__main__.TestCases.test_custom_list)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\lamp6\AppData\Local\Temp\tmpji9x9zvl.py", line 32, in test_custom_list
    self.assertGreater(result, 0)
    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^
AssertionError: 0.0 not greater than 0

FAIL: test_